In [ ]:
from data import PAD_TOKEN, START_EN_TOKEN, START_YUE_TOKEN, END_TOKEN, PAD_ID, encode
from utils import ByteSeq
import pickle

with open("shared.pkl", "rb") as f:
    shared = pickle.load(f)

en_chars = yue_chars = shared["chars"]
en_tokens = yue_tokens = shared["tokens"]

vocab_list = [PAD_TOKEN] + en_tokens + sorted(en_chars) + [START_EN_TOKEN, START_YUE_TOKEN, END_TOKEN]
vocab = {token: i for i, token in enumerate(vocab_list)}
merges = en_tokens

In [ ]:
from data import TranslationDataset, collate_fn
from torch.utils.data import DataLoader, random_split


dataset = TranslationDataset("en.txt", "yue.txt", vocab, merges)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_set, batch_size=128, collate_fn=collate_fn)
test_loader = DataLoader(test_set, batch_size=128, collate_fn=collate_fn)

In [ ]:
D_MODEL = 256
N_HEADS = 8
N_LAYERS = 3
DROPOUT = 0.2

model = Transformer(
    source_vocab_size=len(vocab),
    target_vocab_size=len(vocab),
    d_model=D_MODEL,
    n_heads=N_HEADS,
    source_l_layers=N_LAYERS,
    target_l_layers=N_LAYERS,
    max_seq_len=512,
    dropout=DROPOUT
)

device = torch.device("cuda")
model = model.to(device)

WARMUP_STEPS = 4000
optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)

def lr_lambda(step):
    step = max(step, 1)
    return D_MODEL ** (-0.5) * min(step ** (-0.5), step * WARMUP_STEPS ** (-1.5))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)

EPOCHS = 25
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for src, tgt in tqdm(train_loader):
        src, tgt = src.to(device), tgt.to(device)
        tgt_input = tgt[:, :-1]
        tgt_label = tgt[:, 1:]

        logits = model(src, tgt_input)
        loss = loss_fn(logits.reshape(-1, len(vocab)), tgt_label.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(device), tgt.to(device)
            tgt_input = tgt[:, :-1] # mask out last token for decoder input
            tgt_label = tgt[:, 1:] # don't need start token for label
            logits = model(src, tgt_input)
            val_loss += loss_fn(logits.reshape(-1, len(vocab)), tgt_label.reshape(-1)).item()

    print(f"Epoch {epoch+1} | Train loss: {total_loss/len(train_loader):.4f} | Val loss: {val_loss/len(val_loader):.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")